In [1]:
import json
import numpy as np
import pandas as pd
from sklearn.metrics import cohen_kappa_score
from statsmodels.stats.inter_rater import fleiss_kappa
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)


In [2]:
def load_annotations(file_path):
    """Load annotations from a JSON file."""
    with open(file_path, 'r', encoding='utf-8') as file:
        return json.load(file)


def parse_nlp_annotations(annotations1, annotations2):
    """Parse NLP annotations by matching based on text content and extracting POS tags."""
    annotations_dict1 = {annotation['id']: annotation for annotation in annotations1}
    annotations_dict2 = {annotation['id']: annotation for annotation in annotations2}
    common_ids = set(annotations_dict1.keys()) & set(annotations_dict2.keys())

    pos_tags1, pos_tags2 = [], []

    for common_id in common_ids:
        labels1 = {label['text'].strip(): label for label in annotations_dict1[common_id]['label']}
        labels2 = {label['text'].strip(): label for label in annotations_dict2[common_id]['label']}

        # Compare based on normalized text keys
        for text, label1 in labels1.items():
            if text in labels2:
                label2 = labels2[text]
                pos_tags1.append(label1['labels'][0])
                pos_tags2.append(label2['labels'][0])
            else:
                print(f"Warning: No match for text '{text}' in ID {common_id}. Skipping.")

    return pos_tags1, pos_tags2


def calculate_cohen_kappa(pos_tags1, pos_tags2):
    """Calculate Cohen's Kappa score and interpret the agreement."""
    # Print unique labels and counts
    all_labels = set(pos_tags1 + pos_tags2)
    print("Number of unique labels:", len(all_labels))
    print("Labels used:", all_labels)

    # Identify mismatches
    mismatches = [(i, tag1, tag2) for i, (tag1, tag2) in enumerate(zip(pos_tags1, pos_tags2)) if tag1 != tag2]
    if mismatches:
        print("\nMismatched Annotations (Annotators Disagree):")
        for mismatch in mismatches:
            print(f"Index: {mismatch[0]}, Annotator 1: {mismatch[1]}, Annotator 2: {mismatch[2]}")
    else:
        print("\nNo mismatches detected. All annotators agree.")

    # Compute Cohen's Kappa
    kappa_score = cohen_kappa_score(pos_tags1, pos_tags2)
    print("Cohen's Kappa:", kappa_score)

    # Interpret the agreement score
    if kappa_score < 0:
        print("No agreement")
    elif kappa_score < 0.2:
        print("Slight agreement")
    elif kappa_score < 0.4:
        print("Fair agreement")
    elif kappa_score < 0.6:
        print("Moderate agreement")
    elif kappa_score < 0.8:
        print("Substantial agreement")
    else:
        print("Almost perfect agreement")

    return kappa_score


def parse_cv_annotations(file_data):
    """Parse CV annotations and extract image name and label."""
    extracted_data = {}
    for item in file_data:
        image_name = item['image'].split('-')[-1]  # Extracts 'img_{number}.jpg'
        label = item['choice']
        extracted_data[image_name] = label
    return extracted_data


def combine_annotations(data1_parsed, data2_parsed, data3_parsed):
    """Combine annotations by image."""
    combined_data = {}
    for image_name in set(data1_parsed.keys()).union(data2_parsed.keys()).union(data3_parsed.keys()):
        combined_data[image_name] = []
        if image_name in data1_parsed:
            combined_data[image_name].append(data1_parsed[image_name])
        if image_name in data2_parsed:
            combined_data[image_name].append(data2_parsed[image_name])
        if image_name in data3_parsed:
            combined_data[image_name].append(data3_parsed[image_name])
    return combined_data


def calculate_fleiss_kappa(combined_data):
    """Calculate Fleiss Kappa score and interpret the agreement."""
    df = pd.DataFrame.from_dict(combined_data, orient='index')
    mismatches = df[df.nunique(axis=1) > 1]
    all_labels = set(df.values.flatten())
    print("Number of unique labels:", len(all_labels))
    print("Labels used:", all_labels)

    if not mismatches.empty:
        print("\nMismatched Annotations (Annotators Disagree):")
        print(mismatches)
    else:
        print("\nNo mismatches detected. All annotators agree.")

    label_map = {label: i for i, label in enumerate(all_labels)}
    df_mapped = df.applymap(lambda x: label_map[x])
    rating_table = []
    for row in df_mapped.itertuples(index=False):
        counts = [0] * len(label_map)
        for label in row:
            counts[label] += 1
        rating_table.append(counts)

    fleiss_kappa_score = fleiss_kappa(rating_table)
    print("\nFleiss Kappa Score:", fleiss_kappa_score)
    if 0.80 <= fleiss_kappa_score <= 1.00:
        print("Very good agreement")
    elif 0.60 <= fleiss_kappa_score < 0.80:
        print("Good agreement")
    elif 0.40 <= fleiss_kappa_score < 0.60:
        print("Moderate agreement")
    elif 0.20 <= fleiss_kappa_score < 0.40:
        print("Fair agreement")
    else:
        print("Poor agreement")

    return fleiss_kappa_score


def summary(kappa_score, fleiss_kappa_score):
    """Print a summary of Cohen's Kappa and Fleiss Kappa scores."""
    print("\nSummary:")
    print(f"Cohen's Kappa Score: {kappa_score}")
    print(f"Fleiss Kappa Score: {fleiss_kappa_score}")



In [3]:

# Main execution
annotations1 = load_annotations('NLP_23110065.json')
annotations2 = load_annotations('NLP_23110066.json')
pos_tags1, pos_tags2 = parse_nlp_annotations(annotations1, annotations2)
kappa_score = calculate_cohen_kappa(pos_tags1, pos_tags2)
print("-" * 130)

data1 = load_annotations('CV_23110065.json')
data2 = load_annotations('CV_23110066.json')
data3 = load_annotations('CV_third-member.json')
data1_parsed = parse_cv_annotations(data1)
data2_parsed = parse_cv_annotations(data2)
data3_parsed = parse_cv_annotations(data3)
combined_data = combine_annotations(data1_parsed, data2_parsed, data3_parsed)
fleiss_kappa_score = calculate_fleiss_kappa(combined_data)

summary(kappa_score, fleiss_kappa_score)


Number of unique labels: 14
Labels used: {'VERB', 'DET', 'NUM', 'ADP', 'X', 'PRON', 'PART_NEG', 'CONJ', 'PART', 'NOUN', 'ADV', 'PROPN', 'ADJ', 'PRON_WH'}

Mismatched Annotations (Annotators Disagree):
Index: 38, Annotator 1: ADV, Annotator 2: PART
Index: 60, Annotator 1: NOUN, Annotator 2: PROPN
Index: 68, Annotator 1: NUM, Annotator 2: NOUN
Index: 69, Annotator 1: PRON_WH, Annotator 2: NOUN
Index: 78, Annotator 1: PROPN, Annotator 2: NOUN
Index: 91, Annotator 1: PART, Annotator 2: ADP
Index: 93, Annotator 1: ADJ, Annotator 2: PART
Index: 97, Annotator 1: NOUN, Annotator 2: PROPN
Index: 148, Annotator 1: ADV, Annotator 2: ADP
Index: 157, Annotator 1: NOUN, Annotator 2: ADJ
Index: 222, Annotator 1: ADP, Annotator 2: ADJ
Index: 230, Annotator 1: PRON, Annotator 2: ADJ
Index: 232, Annotator 1: PRON, Annotator 2: PROPN
Index: 260, Annotator 1: NOUN, Annotator 2: PROPN
Index: 265, Annotator 1: VERB, Annotator 2: ADJ
Index: 286, Annotator 1: PROPN, Annotator 2: NOUN
Index: 292, Annotator 1: 